<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Cuaderno 00 · Conoce la empresa y sus datos
**Excel Corporativo · 12 Claves** — Recorrido visual por Distribuidora Andina · ⏱ 20 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/00_conoce_la_empresa.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#empresa)

<sub>Si al abrir un enlace Google muestra «Redirect Notice», haz clic en el enlace azul de esa página: es un aviso de seguridad de Google, no un error.</sub>

**Al terminar podrás:**
1. Explicar qué hace la empresa y qué área produce cada tabla.
2. Responder la pregunta clave ante cualquier tabla: ¿qué representa una fila?
3. Leer los tres gráficos básicos de la empresa.
4. Comprobar que tu laboratorio funciona (Taller 0).

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden. Si no lo haces, no pasa nada grave: el original está protegido y nadie puede dañarlo desde aquí, pero lo que cambies se borra al cerrar.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · La empresa en un minuto
**Distribuidora Andina S.A.S.** vende suministros de oficina y tecnología a otras organizaciones: pymes, empresas corporativas, entidades de gobierno e instituciones educativas. Tiene seis sedes (Bogotá, Medellín, Cali, Barranquilla, Bucaramanga y Pereira), cinco líneas de producto y tres canales de venta: presencial, televentas y e-commerce.

Cada tabla la produce un área. **Comercial** registra los pedidos; **Facturación** emite y anula facturas; **Cartera** registra el recaudo; **Servicio al cliente** crea los clientes; **Compras** mantiene precios y costos; **Talento Humano** registra a los vendedores; **Planeación financiera** fija las metas, y **TI** exporta los datos del ERP.

[Ver la explicación completa en el tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#empresa)

## 2 · ¿Qué representa una fila?
Es la primera pregunta ante cualquier tabla. En `ventas`, **una fila es una línea de factura**: un producto dentro de una factura. Mira esta factura real de tres productos y cómo queda guardada.

In [ ]:
# @title Ver una factura y sus filas
det = df[(df.estado == "Pagada") & (df.descuento_pct > 0)]
conteo = det.groupby("id_factura").size()
factura = conteo[conteo == 3].index[5]
filas = df[df.id_factura == factura].sort_values("id_linea")
f0 = filas.iloc[0]
display(Markdown(f"**Factura {factura}** · {f0.fecha:%Y-%m-%d} · {f0.razon_social} ({f0.segmento}) · sede {f0.ciudad} · vendedor {f0.nombre_vendedor}"))
vista = filas[["id_linea", "id_factura", "id_cliente", "sku", "producto", "cantidad", "precio_unitario", "subtotal", "iva", "total", "estado"]]
display(vista.style.format({"precio_unitario": pesos, "subtotal": pesos, "iva": pesos, "total": pesos}).hide(axis="index"))
display(Markdown(f"La factura completa vale **{pesos(filas.total.sum())}** con IVA. Ese total no se guarda en ninguna fila: se calcula sumando las {len(filas)} filas."))

## 3 · Las tablas del conjunto de datos
Cada tabla responde a la pregunta «¿qué representa una fila?» de forma distinta. Esa respuesta se llama **granularidad**.

In [ ]:
# @title Ver el resumen de tablas
resumen = pd.DataFrame([
    ["ventas", "Una línea de factura", len(ventas), ventas.shape[1], "id_linea", "Comercial y Facturación"],
    ["clientes", "Un cliente", len(clientes), clientes.shape[1], "id_cliente", "Servicio al cliente"],
    ["productos", "Un producto", len(productos), productos.shape[1], "sku", "Compras"],
    ["vendedores", "Un vendedor", len(vendedores), vendedores.shape[1], "id_vendedor", "Talento Humano"],
    ["sedes", "Una sede", len(sedes), sedes.shape[1], "id_sede", "Gerencia"],
    ["metas", "La meta de una sede en un mes", len(metas), metas.shape[1], "id_sede + anio + mes", "Planeación financiera"],
], columns=["Tabla", "Una fila es…", "Filas", "Columnas", "Llave", "Área responsable"])
display(resumen.style.format({"Filas": lambda x: f"{x:,}".replace(",", ".")}).hide(axis="index"))

## 4 · Tres gráficos para conocer el negocio

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo se comporta la venta durante el año? |
> | **Qué se grafica** | Eje X: mes · Eje Y: venta neta (millones) · Color: año · Sin anuladas |
> | **Por qué esta forma** | El tiempo va en el eje horizontal y una línea muestra la tendencia mes a mes. |
> | **Qué se descartó** | Columnas por mes: 24 barras saturan el gráfico y ocultan la forma del año. |
> | **Cómo se lee** | Busca los picos (temporada escolar y cierre de año) y compara la altura de las dos líneas. |
> | **En Excel 365** | Tabla dinámica con mes en Filas y año en Columnas › Gráfico dinámico de líneas. |

In [ ]:
# @title Gráfico 1 · Venta mensual
mensual = validas.groupby(["anio", "mes"], as_index=False)["subtotal"].sum()
mensual["millones"] = mensual["subtotal"] / 1e6
mensual["año"] = mensual["anio"].astype(str)
top = mensual[mensual.anio == 2025].sort_values("subtotal").iloc[-1]
fig = px.line(mensual, x="mes", y="millones", color="año", markers=True,
              color_discrete_map={"2024": GRIS, "2025": AZUL},
              title=f"En 2025 el mes más fuerte fue {MESES[int(top.mes) - 1].lower()}: {millones(top.subtotal)}")
fig.update_xaxes(tickvals=list(range(1, 13)), ticktext=MESES, title="")
fig.update_layout(yaxis_title=MM, legend_title="", hovermode="x unified")
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué sede pesa más? |
> | **Qué se grafica** | Barras: una por sede · Largo: la medida que elijas · Año: el que elijas |
> | **Por qué esta forma** | Barras horizontales ordenadas: se lee el ranking de un vistazo y caben los nombres. |
> | **Qué se descartó** | Torta: seis partes parecidas no se comparan bien por ángulos. |
> | **Cómo se lee** | La barra resaltada es la primera. Compara su largo con las demás. |
> | **En Excel 365** | Tabla dinámica con ciudad en Filas, ordenada de mayor a menor › Barra agrupada. |

In [ ]:
# @title Gráfico 2 · Ranking de sedes
medida = "Venta neta"  # @param ["Venta neta", "Margen bruto", "Facturas"]
anio = "2025"  # @param ["2025", "2024"]
base = validas[validas.anio == int(anio)]
if medida == "Facturas":
    serie = base.groupby("ciudad")["id_factura"].nunique(); etiqueta = serie.map(lambda v: f"{v:,}".replace(",", "."))
else:
    col = "subtotal" if medida == "Venta neta" else "margen_bruto"
    serie = base.groupby("ciudad")[col].sum() / 1e6; etiqueta = serie.map(lambda v: f"$ {v:,.0f} M".replace(",", "."))
serie = serie.sort_values()
colores = [GRIS] * (len(serie) - 1) + [AZUL]
fig = go.Figure(go.Bar(x=serie.values, y=serie.index, orientation="h", marker_color=colores, text=etiqueta[serie.index], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"{serie.index[-1]} lidera en {medida.lower()} {anio} con el {porc(serie.iloc[-1] / serie.sum())} del total",
                  xaxis_title=MM if medida != "Facturas" else "Facturas", height=380)
dar_aire(fig, serie.max())
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué líneas venden más y cuáles dejan más margen? |
> | **Qué se grafica** | Columnas agrupadas: venta y margen por línea, 2025 |
> | **Por qué esta forma** | Dos medidas en la misma unidad (pesos) comparadas lado a lado en cada línea. |
> | **Qué se descartó** | Dos ejes: si venta y margen tuvieran escalas distintas, el gráfico engañaría. |
> | **Cómo se lee** | Compara la altura de la barra de margen con la de venta en cada línea: tecnología vende mucho y deja poco. |
> | **En Excel 365** | Tabla dinámica con categoría en Filas y subtotal y margen_bruto en Valores › Columna agrupada. |

In [ ]:
# @title Gráfico 3 · Venta y margen por línea
lin = validas[validas.anio == 2025].groupby("categoria")[["subtotal", "margen_bruto"]].sum().sort_values("subtotal", ascending=False) / 1e6
largo = lin.reset_index().melt(id_vars="categoria", var_name="medida", value_name="millones")
largo["medida"] = largo["medida"].map({"subtotal": "Venta neta", "margen_bruto": "Margen bruto"})
m = (lin["margen_bruto"] / lin["subtotal"])
fig = px.bar(largo, x="categoria", y="millones", color="medida", barmode="group", color_discrete_sequence=[AZUL, TEAL],
             title=f"{lin.index[0]} es la línea que más vende, pero su margen es el más bajo ({porc(m[lin.index[0]])})" if m.idxmin() == lin.index[0]
                   else f"{lin.index[0]} es la línea que más vende en 2025")
fig.update_layout(yaxis_title=MM, xaxis_title="", legend_title="")
fig.show()

## 5 · Explora una columna
Elige una columna y mira qué valores tiene y cuánto pesa cada uno. Así se conoce una tabla antes de analizarla.

In [ ]:
# @title Explorar los valores de una columna
columna = "canal"  # @param ["canal", "segmento", "categoria", "estado", "medio_pago", "ciudad"]
tabla = df.groupby(columna).agg(lineas=("id_linea", "size"), venta=("subtotal", "sum")).sort_values("venta", ascending=False)
tabla["% de la venta"] = tabla["venta"] / tabla["venta"].sum()
display(tabla.style.format({"lineas": lambda x: f"{x:,}".replace(",", "."), "venta": pesos, "% de la venta": porc}))
if columna == "estado":
    display(Markdown("**Ojo:** las filas *Anulada* existen en la tabla, pero no son venta. En los análisis comerciales se excluyen."))

## 6 · Retos de interpretación
Elige la respuesta en la lista desplegable y ejecuta la celda con ▶.

In [ ]:
# @title Reto 1 · La factura de ejemplo tiene 3 productos. ¿Cuántas filas ocupa en la tabla ventas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "1", "3", "4", "Depende del IVA"]
revisar(respuesta, "3", """Una fila por producto: la granularidad de ventas es la línea de factura.""")

In [ ]:
# @title Reto 2 · ¿Qué mes de 2025 tuvo la mayor venta neta?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Ene", "Jun", "Sep", "Dic"]
revisar(respuesta, MESES[validas[validas.anio == 2025].groupby("mes")["subtotal"].sum().idxmax() - 1], """Es el punto más alto de la línea azul del gráfico 1.""")

In [ ]:
# @title Reto 3 · ¿Qué área produce la tabla metas?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Comercial", "Planeación financiera", "Talento Humano", "TI"]
revisar(respuesta, "Planeación financiera", """Planeación financiera fija el presupuesto de cada sede, mes a mes.""")

In [ ]:
# @title Comprobación del laboratorio (Taller 0)
ok = len(ventas) > 17000 and len(metas) == 144 and abs(validas[validas.anio == 2025].subtotal.sum() - 14_692_379_210) < 1
display(Markdown("## ✅ Laboratorio listo" if ok else "## ❌ Algo no cargó bien: vuelve a ejecutar todas las celdas"))
display(Markdown(f"Leíste {miles(len(ventas))} líneas de venta. Venta neta 2025 sin anuladas: **{pesos(validas[validas.anio == 2025].subtotal.sum())}**. "
                 "Si ves este mensaje y el gráfico de abajo, tu entorno funciona."))
mini = validas.groupby(validas.fecha.dt.to_period("M"))["subtotal"].sum() / 1e6
fig = px.line(x=mini.index.to_timestamp(), y=mini.values, title="Venta neta mensual 2024–2025", markers=True)
fig.update_layout(height=300, xaxis_title="", yaxis_title=MM)
eje_meses(fig, mini.index.to_timestamp())
fig.show()

## Resumen
* Una fila de ventas es una línea de factura: contar filas no es contar facturas.
* Cada tabla la produce un área; conocerla explica sus errores típicos.
* Antes de analizar, explora qué valores tiene cada columna.

**Siguiente:** [Clave 1 · Estructura antes que fórmulas](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-01) · [Taller 1](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-1)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>